In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pycolmap
import numpy as np
import cv2

In [ ]:

import sys
sys.path.append("../src/")

In [ ]:
from drone_core import get_poses_from_data, read_images_data_from_folder
from colmap_processing import update_db_pose_prior

## Read the images data and metadata poses

In [ ]:
src_data_dir = "G:/Work/AI/NeRF/drone_dataset/raw_data/dataset_testing/picnic_bench/images/level_1"
images_data = read_images_data_from_folder(src_data_dir)
sample_image = cv2.imread(f"{src_data_dir}/{images_data[0]['file_name']}")
H, W = sample_image.shape[:2]
print(f"Number of images found = {len(images_data)} with image resolution H = {H} and W = {W}")

To use the COLMAP soft prior feature The sequence using COLMAP CLI would be as follows

```bash
colmap feature_extractor \
--database_path <db_path> \
--image_path <image_path> \
--ImageReader.single_camera 1 \
--ImageReader.camera_model OPENCV
```

```bash
colmap exhaustive_matcher \
--database_path <db path>
```

--> Update the database with the required soft prior and variance (as shown in this notebook)

```bash
colmap pose_prior_mapper \
--database_path <db path> \
--image_path <image path> \
--output_path <optimized model
```

Note: since the images have the gps data in the metadata colmap will read this information during feature extraction phase but will set the variance to non valid value so the priors wont be usable, if you need to use the priors as gps formate (WGS84) u can set the variance from the command line as follows

```bash
colmap pose_prior_mapper \
--database_path <db path> \
--image_path <image path> \
--output_path <optimized model> \
--prior_position_std_x <var_value> \
--prior_position_std_y <var_value> \
--prior_position_std_z <var_value> \
--overwrite_priors_covariance 1
```

## Set COLMAP soft prior

In [ ]:
colmap_data_base = "G:/Work/AI/NeRF/drone_dataset/processed/bench/COLMAP_PRIOR/database.db"
cov_x = 4
cov_y = 4
cov_z = 4

In [ ]:
update_db_pose_prior(colmap_data_base, images_data, [cov_x, cov_y, cov_z],
                     update_position=True, cartesian_system=True)